# Process Fugro CSV Files

This notebook converts raw Fugro CSV files into standardized per-series CSV files with consistent column names:
- First column (index): `Time` (datetime)
- Second column (data): `head` (numeric)

Output files are saved to `output_data/only_csv_fugro/`

In [1]:
# Helper functions and setup
from pathlib import Path
import re
import pandas as pd


def find_repo_root(start=Path.cwd()):
    """Return the first ancestor (including start) that contains .git or pyproject.toml."""
    p = start.resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / '.git').exists() or (candidate / 'pyproject.toml').exists():
            return candidate
    return p


def sanitize_for_filename(name: str) -> str:
    """
    Make a string safe for filenames on Windows/Linux/macOS by replacing
    unwanted characters with underscores and collapsing repeats.
    """
    safe = re.sub(r'[^0-9A-Za-z._-]+', '_', str(name))
    safe = safe.strip(' ._')
    return safe or "series"


repo_root = find_repo_root()
print('Repository root detected as:', repo_root)

Repository root detected as: D:\Users\jvanruitenbeek\data_validation


In [2]:
# Processing Function

In [3]:
def process_fugro_csv(input_file, repo_root, dayfirst=True, drop_all_nan=True):
    """
    Process a Fugro-style CSV (including Vista Data Vision format):

    Args:
        input_file (str or Path): Path to input Fugro CSV file
        repo_root (str or Path): Repository root path
        dayfirst (bool): Whether dates are in D-M-Y format (default: True for European format)
        drop_all_nan (bool): Skip columns that are entirely NaN (default: True)
    
    Returns:
        list: Paths to all saved CSV files
    """

    input_path = Path(input_file)
    repo_root = Path(repo_root)

    origin_stem = input_path.stem
    out_dir = repo_root / "output_data" / "fugro" / origin_stem / "only_csv"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Detect Vista Data Vision format
    skiprows = 0
    try:
        with open(input_path, "r", encoding="utf-8-sig", errors="replace") as f:
            first_line = f.readline().strip()
            if "Vista Data Vision" in first_line:
                skiprows = 5  # skip 5 lines so the header row (line 6) is used as column names
                print(f"Detected Vista Data Vision file -> skipping {skiprows} header rows")
    except Exception:
        pass

    # Read raw CSV
    df = pd.read_csv(
        input_path,
        sep=",",
        header=0,
        dtype="object",
        encoding="utf-8-sig",
        engine="python",
        skiprows=skiprows,
    )

    # Normalize first column to "Time"
    first_col = str(df.columns[0]).replace("\ufeff", "").strip()
    if first_col.lower() != "time":
        df.rename(columns={df.columns[0]: "Time"}, inplace=True)

    # ------------------------------------------------------------------
    #      ROBUST TIMESTAMP PARSING WITH AUTO-DETECTION
    # ------------------------------------------------------------------
    time_raw = df["Time"].astype(str)

    # Try ISO format (YYYY-MM-DD)
    dt_iso = pd.to_datetime(time_raw, format="%Y-%m-%d %H:%M:%S", errors="coerce")

    # Try flexible dayfirst parsing
    dt_dayfirst = pd.to_datetime(time_raw, dayfirst=dayfirst, errors="coerce")

    # Pick the parsing that yields MORE valid timestamps
    if dt_iso.notna().sum() >= dt_dayfirst.notna().sum():
        df["Time"] = dt_iso
    else:
        df["Time"] = dt_dayfirst

    # Remove invalid timestamps & set index
    df = df.dropna(subset=["Time"]).set_index("Time")

    # ------------------------------------------------------------------
    # Save each column as its own head-series CSV
    # ------------------------------------------------------------------
    written = []
    seen_names = {}

    for col in df.columns:
        ser = pd.to_numeric(df[col], errors="coerce")

        if drop_all_nan and ser.notna().sum() == 0:
            continue

        base_name = sanitize_for_filename(col)
        count = seen_names.get(base_name, 0)
        out_name = base_name if count == 0 else f"{base_name}_{count}"
        seen_names[base_name] = count + 1

        out_df = pd.DataFrame({"head": ser})

        out_path = out_dir / f"{out_name}.csv"
        out_df.to_csv(out_path, index=True, index_label="Time")

        print(f"Saved {out_path} ({ser.notna().sum()} rows)")

        written.append(out_path)

    print(f"Done. Saved {len(written)} series to {out_dir}")
    return written

In [7]:
# List all available Fugro CSV files
fugro_dir = repo_root / 'input_data' / 'Fugro'
files = sorted(fugro_dir.glob('*.csv'))
print(f'Found {len(files)} CSV file(s) in {fugro_dir.name}:\n')
for i, f in enumerate(files, 1):
    print(f'{i:2d}. {f.name}')


file_to_process = 3

# Process a specific file
if len(files) > 0:
    print(f'\nProcessing: {files[file_to_process].name}')
    process_fugro_csv(
        input_file=files[file_to_process],
        repo_root=repo_root
    )

Found 5 CSV file(s) in Fugro:

 1. 4423-241417_PB_HHW_01-01-2023 00_00_00_29-06-2026 00_00_00_Uur_20260629115503.csv
 2. 4423-241417_PB_HOORN_01-01-2023 00_00_00_29-06-2026 00_00_00_Uur_20260629115702.csv
 3. 4424-260484_HHW_normaal_01-01-2023 00_00_00_29-06-2026 00_00_00_Uur_20260629115831.csv
 4. 4424_Hoorn_Zuiderdijk_01-01-2023 00_00_00_29-06-2026 00_00_00_Uur_20260629115724.csv
 5. NL-241038-API_GTA11653.csv

Processing: 4424_Hoorn_Zuiderdijk_01-01-2023 00_00_00_29-06-2026 00_00_00_Uur_20260629115724.csv
Detected Vista Data Vision file -> skipping 5 header rows


C:\Users\jvanruitenbeek\AppData\Local\Temp\70\ipykernel_14104\4202381686.py:58: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt_dayfirst = pd.to_datetime(time_raw, dayfirst=dayfirst, errors="coerce")


Saved D:\Users\jvanruitenbeek\data_validation\output_data\fugro\4424_Hoorn_Zuiderdijk_01-01-2023 00_00_00_29-06-2026 00_00_00_Uur_20260629115724\only_csv\NL-263591-FB-FLB5029_B09PB01_-7.1_-8.1_m_NAP_avg.csv (13475 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\fugro\4424_Hoorn_Zuiderdijk_01-01-2023 00_00_00_29-06-2026 00_00_00_Uur_20260629115724\only_csv\NL-263591-FB-FLB5029_B09PB02_-2.3_-3.3_m_NAP_avg.csv (13475 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\fugro\4424_Hoorn_Zuiderdijk_01-01-2023 00_00_00_29-06-2026 00_00_00_Uur_20260629115724\only_csv\NL-263591-FB-FLB5036_B08PB01_-5.0_-6.0_m_NAP_avg.csv (13452 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\fugro\4424_Hoorn_Zuiderdijk_01-01-2023 00_00_00_29-06-2026 00_00_00_Uur_20260629115724\only_csv\NL-263591-FB-FLB5036_B08PB02_-1.4_-2.4_m_NAP_avg.csv (13452 rows)
Saved D:\Users\jvanruitenbeek\data_validation\output_data\fugro\4424_Hoorn_Zuiderdijk_01-01-2023 00_00_00_29-06-2026 00_